In [3]:
!nvidia-smi

Mon Jun  1 04:40:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:

%%writefile gpu_tictactoe.cu
#include <iostream>
#include <cuda_runtime.h>

using namespace std;

#define EMPTY 0
#define X 1
#define O 2

void printBoard(int board[9])
{
    cout << endl;

    for(int i=0;i<9;i++)
    {
        char c='-';

        if(board[i]==X) c='X';
        if(board[i]==O) c='O';

        cout << c << " ";

        if((i+1)%3==0)
            cout << endl;
    }

    cout << endl;
}

int checkWinner(int board[9])
{
    int wins[8][3]={
        {0,1,2},{3,4,5},{6,7,8},
        {0,3,6},{1,4,7},{2,5,8},
        {0,4,8},{2,4,6}
    };

    for(int i=0;i<8;i++)
    {
        int a=wins[i][0];
        int b=wins[i][1];
        int c=wins[i][2];

        if(board[a]!=0 &&
           board[a]==board[b] &&
           board[b]==board[c])
            return board[a];
    }

    return 0;
}

bool draw(int board[9])
{
    for(int i=0;i<9;i++)
    {
        if(board[i]==0)
            return false;
    }

    return true;
}

// GPU 1: choose first empty cell
__global__ void gpu1Move(int *board,int *move)
{
    int tid=threadIdx.x;

    if(tid<9 && board[tid]==EMPTY)
    {
        atomicMin(move,tid);
    }
}

// GPU 2: prefer center, else first empty
__global__ void gpu2Move(int *board,int *move)
{
    if(threadIdx.x==0)
    {
        if(board[4]==EMPTY)
        {
            *move=4;
            return;
        }

        for(int i=0;i<9;i++)
        {
            if(board[i]==EMPTY)
            {
                *move=i;
                return;
            }
        }
    }
}

int main()
{
    int board[9]={0};

    int *d_board;
    int *d_move;

    cudaMalloc(&d_board,9*sizeof(int));
    cudaMalloc(&d_move,sizeof(int));

    int turn=X;

    cout<<"======================"<<endl;
    cout<<" GPU vs GPU TicTacToe "<<endl;
    cout<<"======================"<<endl;

    while(true)
    {
        cudaMemcpy(
            d_board,
            board,
            9*sizeof(int),
            cudaMemcpyHostToDevice);

        int move=100;

        cudaMemcpy(
            d_move,
            &move,
            sizeof(int),
            cudaMemcpyHostToDevice);

        if(turn==X)
        {
            gpu1Move<<<1,9>>>(d_board,d_move);
        }
        else
        {
            gpu2Move<<<1,1>>>(d_board,d_move);
        }

        cudaDeviceSynchronize();

        cudaMemcpy(
            &move,
            d_move,
            sizeof(int),
            cudaMemcpyDeviceToHost);

        board[move]=turn;

        if(turn==X)
            cout<<"GPU 1 (X) move: "<<move<<endl;
        else
            cout<<"GPU 2 (O) move: "<<move<<endl;

        printBoard(board);

        int winner=checkWinner(board);

        if(winner==X)
        {
            cout<<"GPU 1 WINS"<<endl;
            break;
        }

        if(winner==O)
        {
            cout<<"GPU 2 WINS"<<endl;
            break;
        }

        if(draw(board))
        {
            cout<<"DRAW GAME"<<endl;
            break;
        }

        turn=(turn==X)?O:X;
    }

    cudaFree(d_board);
    cudaFree(d_move);

    return 0;
}

Writing gpu_tictactoe.cu


In [5]:
!nvcc gpu_tictactoe.cu -o game

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [6]:
!./game

 GPU vs GPU TicTacToe 
GPU 1 (X) move: 0

X - - 
- - - 
- - - 

GPU 2 (O) move: 4

X - - 
- O - 
- - - 

GPU 1 (X) move: 1

X X - 
- O - 
- - - 

GPU 2 (O) move: 2

X X O 
- O - 
- - - 

GPU 1 (X) move: 3

X X O 
X O - 
- - - 

GPU 2 (O) move: 5

X X O 
X O O 
- - - 

GPU 1 (X) move: 6

X X O 
X O O 
X - - 

GPU 1 WINS
